# 演習3 解答編 ―― `mutex` で守る

> まず `ex03_mutex.ipynb` を自分で解いてから読んでください。

## 発展課題1 の解答 ―― ループ全体をロックする

```cpp
while (true) {
    std::lock_guard<std::mutex> guard(mtx);
    if (q.empty()) return;
    q.pop();
    taken[id]++;
}
```

- **正しく動くか** ⇒ 動きます。合計は必ず 200,000 になります。
  `lock_guard` は `while` の**中**で作られているので、1周ごとに施錠・解錠が繰り返されます
  （`{ }` を書かなくても、`while` の本体そのものがスコープです）。
- **速いか** ⇒ この例ではほとんど変わりません。`q.pop()` も `taken[id]++` も一瞬で終わるからです。
  ただし `taken[id]++` まで鍵の中に入っているのは無駄で、`ex03b.cpp` のように
  鍵の中の処理が重くなった瞬間に効いてきます。

### `lock_guard` を `while` の「外」に置いたら

ここが本題です。**`{ }` の位置が1つずれるだけで、意味がまったく変わります。**

```cpp
void worker(int id) {
    std::lock_guard<std::mutex> guard(mtx);   // ← while に入る前に施錠
    while (true) { ... }                       //    解錠されるのは worker を抜けるとき
}
```

`lock_guard` が解錠するのは**スコープを抜けるとき**でした。
`while` の外に置くと、そのスコープは **`worker` 関数まるごと**です。
つまり **200,000件を処理し終えるまで、一度も鍵を離しません。**

図にすると、こうなります（`L`＝施錠、`U`＝解錠、数字＝1件処理、`.`＝鍵が空くのを待っている）。

```
【while の中に置いた場合（正しい版）】
T0    L1U L1U L1U ...   L1U      交互に取り合う
T1    L1U L1U L1U ...   L1U      → 内訳はおよそ半々

【while の外に置いた場合】
T0    L11111111111111111111 ... 1111111U      ← 200,000件すべてを1人で処理
T1    ..................................L(空)U ← 鍵が空いたときには、もうキューも空
```

デッドロックはしません。**プログラムは正常に終わり、合計も正しく 200,000 になります。**
しかし内訳は `200000 / 0`。スレッドを2本立てた意味が完全に消えています。

> **さらに一歩**：`while` の外と中の**両方**に `lock_guard` を置くとどうなるでしょうか。
> 今度は同じスレッドが同じ鍵を二重にロックしようとして、**本当にデッドロックします**。
> `std::mutex` は「一度に1スレッドしか通さない」だけでなく、
> **同じスレッドからの2回目のロックも通しません**（自分で自分を締め出す形になります）。

2つとも確かめてみましょう。まず「`while` の外に置いた版」です。


In [ ]:
%%writefile ans03a.cpp
#include <iostream>
#include <thread>
#include <queue>
#include <mutex>

std::queue<int> q;
std::mutex mtx;
long taken[2] = {0, 0};

void worker(int id) {
    std::lock_guard<std::mutex> guard(mtx);     // ← while の「外」で施錠
    while (true) {
        if (q.empty()) return;
        q.pop();
        taken[id]++;
    }
}

int main() {
    for (int i = 0; i < 200000; i++) q.push(i);
    std::thread t1(worker, 0), t2(worker, 1);
    t1.join(); t2.join();
    std::cout << "スレッド0 = " << taken[0] << " / スレッド1 = " << taken[1]
              << " / 合計 = " << taken[0] + taken[1] << "\n";
    return 0;
}

In [ ]:
!g++ -std=c++17 -pthread ans03a.cpp -o ans03a
!for i in 1 2 3; do ./ans03a; done

合計は 200,000 で正しいのに、内訳は **`200000 / 0`** または **`0 / 200000`** です
（どちらのスレッドが全部取るかは実行ごとに変わります）。

先に鍵を取ったスレッドが200,000件すべてを1人で処理し、もう片方は最後まで何もできていません。
**答えは正しいのに、並列にした意味が完全に失われています。**

---

### 次は「`while` の外と中の両方に置いた版」

`{ }` を消したうえで、`while` の外にも `lock_guard` を書いてしまった、という形です。

```cpp
void worker(int id) {
    std::lock_guard<std::mutex> outer(mtx);        // ① while の「外」で施錠
    while (true) {
        std::lock_guard<std::mutex> inner(mtx);    // ② while の「中」でもう一度施錠
        if (q.empty()) return;
        q.pop();
        taken[id]++;
    }
}
```

①で鍵をかけたスレッドが、②で**同じ鍵をもう一度かけようとします**。
ここで起きるのは、これまでとは別種の事故です。

> **`std::mutex` は「一度に1スレッドしか通さない」だけでなく、
> 同じスレッドからの2回目のロックも通しません。**

①で鍵をかけたのは自分自身ですが、`mutex` は「誰がかけたか」を区別しません。
**自分でかけた鍵に、自分が締め出される**わけです。
そして鍵を開けられるのは①を抜けたときだけ、①を抜けるには②を通らなければならない ――
永久に抜け出せません。これが**デッドロック**です。

次のセルで、実際にこの `worker` を動かして確かめます。
止まったままにならないよう、5秒で強制終了させます。
（`timeout` は「指定秒数で打ち切る」Linux のコマンドで、
打ち切ったときの終了コードは **124** です。）


In [ ]:
%%writefile ans03b.cpp
#include <iostream>
#include <thread>
#include <queue>
#include <mutex>

std::queue<int> q;
std::mutex mtx;
long taken[2] = {0, 0};

void worker(int id) {
    std::lock_guard<std::mutex> outer(mtx);        // ① while の「外」で施錠
    std::cout << "スレッド" << id << " : 外側で施錠できました\n" << std::flush;

    while (true) {
        std::cout << "スレッド" << id << " : 内側でもう一度施錠します...\n" << std::flush;
        std::lock_guard<std::mutex> inner(mtx);    // ② 同じスレッドが同じ鍵を二重に → ここで止まる
        std::cout << "ここには絶対に到達しません\n";
        if (q.empty()) return;
        q.pop();
        taken[id]++;
    }
}

int main() {
    for (int i = 0; i < 200000; i++) q.push(i);
    std::thread t1(worker, 0), t2(worker, 1);
    t1.join(); t2.join();                          // ここから先へ進めない
    std::cout << "合計 = " << taken[0] + taken[1] << "\n";
    return 0;
}


In [ ]:
!g++ -std=c++17 -pthread ans03b.cpp -o ans03b
!timeout 5 ./ans03b; echo "終了コード=$? （124 なら5秒たっても終わらなかった＝デッドロック）"

「内側でもう一度施錠します...」を表示したまま、**永久に止まります。**

起きていることを整理します。

- 先に ① を通れたスレッド（仮にスレッド0）は、② で自分がかけた鍵に阻まれて止まる
- もう一方のスレッド1は、そもそも ① すら通れず止まっている
- **どちらのスレッドも二度と動かない** ⇒ `main` の `t1.join()` も永久に返らない

**エラーも出ず、異常終了もせず、ただ固まります。** ここがデッドロックのいちばん厄介な点です。
`ans03a.cpp`（`while` の外だけ）は「遅いけれど正しく終わる」バグでしたが、
こちらは「**終わらない**」バグです。`{ }` の位置1つで、この差が出ます。

> **補足**：同じスレッドが同じ鍵を何度でもかけられる `std::recursive_mutex` も用意されています。
> ただし、これが必要になるのは**たいてい設計がこじれている合図**なので、
> まずは「同じ鍵を二重にかけない構造にする」ほうを考えてください。

なお、この形は「うっかり」だけで起きるわけではありません。
**鍵をかけた関数が、内部で別の関数を呼び、その関数がまた同じ鍵をかける**
という間接的な二重ロックが、実際にはいちばんよく起きます。
自分では二重にかけたつもりがないのに止まる、というのが典型的な現れ方です。


## 発展課題2 の解答 ―― `heavy_work` の重さを変える

実測すると、比（広い版 ÷ 狭い版）はおおよそ次のようになります。

```
   WORK        狭い版      広い版       広い/狭い
------------------------------------------------------
     2,000     約   6 ms   約   30 ms   約 5.0 倍
    20,000     約  57 ms   約  140 ms   約 2.4 倍
   200,000     約 560 ms   約 1150 ms   約 2.0 倍
```

**重くするほど 2.0 に近づき、軽くすると 2.0 より大きくなります。**
2つの現象が混ざっているので、順番に見ていきます。

### まず「1件あたり何をしているか」を分解する

```
1件の仕事 ＝ ① キューから取り出す（一瞬）
             ② heavy_work を計算する（WORK に比例。ここが仕事の本体）
             ③ 合計に足す（一瞬）
```

- **狭い版** … 鍵の中は ① と ③ だけ。②は鍵の外なので **2スレッドが本当に同時に**計算できる
- **広い版** … ①②③ が丸ごと鍵の中。②を実行している間ずっと鍵を握っているので、
  **どの瞬間も1スレッドしか計算していない**

式にすると、こうなります。

```
広い版の時間 ≒ 2000件 × ②の時間              ← 実質1スレッド
狭い版の時間 ≒ 2000件 × ②の時間 ÷ 2 ＋ 固定費  ← 2スレッド
```

### なぜ「重くすると 2.0 倍に近づく」のか

`WORK` を大きくすると **② の時間だけが伸びます。** 固定費は増えません。
式の中で②が支配的になるので、比は

```
広い版 ÷ 狭い版 → (2000×②) ÷ (2000×②÷2) = 2.0
```

に落ち着きます。この **2.0 は「スレッド数」そのもの**で、
2スレッドで並列化したときに得られる**理論上の上限**です。
200,000 のときの 2.0 倍は、「並列化が理想どおりに効いている」状態を意味します。

### なぜ「軽くすると 2.0 を超えて 5 倍にもなる」のか

こちらが引っかかりやすいところです。**狭い版が理論以上に速くなったのではありません。
広い版に、`WORK` に比例しない余分な費用が乗っているのです。**

広い版では、片方が鍵を握っている間、もう片方は**鍵が空くのを待つ**しかありません。
待ち時間がある程度以上になると、OS はそのスレッドを**眠らせ**、
鍵が空いたときに**起こし**ます。この「眠る → 起こされる」の往復は
1回あたり数マイクロ秒かかり、これは **`WORK` の大きさとは無関係な固定費**です。

```
WORK = 2,000       … ②は1件あたり数マイクロ秒
                     ＝「眠る/起こす」の費用と同じくらい
                     → 広い版は、仕事と同じだけの費用を余計に払う → 5.0 倍

WORK = 200,000     … ②は1件あたり数百マイクロ秒
                     ＝「眠る/起こす」の費用は誤差の範囲
                     → 余計な費用が見えなくなる → 2.0 倍
```

狭い版では鍵を握っている時間が数ナノ秒しかないため、
もう片方はほとんど待たされず、**眠るところまで行きません**。だから固定費を払わずに済みます。

まとめると、比の読み方はこうなります。

- **比 ≒ 2.0** … 並列化が理論どおりに効いている
- **比 > 2.0** … 広いロックが、計算の遅さに加えて**待ち合わせの費用**まで発生させている
- **比 ≒ 1.0** … そもそも②が軽すぎて、並列化する価値がない

（`WORK=2,000` のときは全体が数msしかなく測定誤差も大きいので、数字は目安として見てください。）

ここから導かれる原則は次のとおりです。

> **鍵の外でできる仕事が多いほど、並列化は効く。**

次のセルで、`heavy_work` の重さを 3 段階に変えて確かめられます。


In [ ]:
%%writefile ans03c.cpp
#include <iostream>
#include <thread>
#include <queue>
#include <mutex>
#include <chrono>
using namespace std::chrono;

std::mutex mtx;
int WORK = 20000;                    // ← 重さ（ex03b.cpp の 20000 に相当）

long heavy_work(int v) {
    long s = 0;
    for (int i = 0; i < WORK; i++) s += (v + i) % 7;
    return s;
}

long run(bool narrow) {
    std::queue<int> q;
    for (int i = 0; i < 2000; i++) q.push(i);
    long total = 0;
    auto body = [&]() {
        while (true) {
            int v;
            if (narrow) {
                { std::lock_guard<std::mutex> g(mtx); if (q.empty()) return; v = q.front(); q.pop(); }
                long r = heavy_work(v);
                { std::lock_guard<std::mutex> g(mtx); total += r; }
            } else {
                std::lock_guard<std::mutex> g(mtx);
                if (q.empty()) return;
                v = q.front(); q.pop();
                total += heavy_work(v);
            }
        }
    };
    auto t0 = steady_clock::now();
    std::thread t1(body), t2(body);
    t1.join(); t2.join();
    return duration_cast<milliseconds>(steady_clock::now() - t0).count();
}

int main() {
    for (int w : {2000, 20000, 200000}) {
        WORK = w;
        long n = run(true), b = run(false);
        std::cout << "WORK=" << w << "\t狭い版 " << n << " ms\t広い版 " << b << " ms"
                  << "\t(広い/狭い = " << (double)b / n << " 倍)\n";
    }
    return 0;
}

In [ ]:
!g++ -std=c++17 -pthread ans03c.cpp -o ans03c && ./ans03c

`WORK` を大きくするほど、比が **2.0（＝スレッド数）に近づいていく**ことが確かめられます。
これが2スレッドで得られる理論上の上限です。

`WORK` が小さいときに 2.0 を超えるのは、**狭い版が理論以上に速いのではなく**、
広い版のほうに「待たされて眠り、起こされる」費用が上乗せされているためでした。

パイプラインで言えば、キューから取り出したあとの**処理そのものは鍵の外**でやります。
鍵を握るのは、キューに1件入れる／出す一瞬だけ。
だから複数の担当者が本当に同時に動けます。


## 発展課題3 の解答 ―― デッドロック

両方のスレッドが永遠に止まります。

```
スレッドA: 鍵1をロック成功 → 鍵2をロックしたい（でもBが持っている）→ 待つ
スレッドB: 鍵2をロック成功 → 鍵1をロックしたい（でもAが持っている）→ 待つ
```

互いに相手が持っている鍵を待ち、どちらも自分の鍵を離さないので、
永久に動きません。これが**デッドロック**です。

プログラムは異常終了せず、エラーも出さず、ただ**固まります**。
競合と違って「たまたま動く」こともあるため、やはり再現が難しいバグです。

古典的な対策は次の2つです。

- **鍵をかける順番を全スレッドで統一する**（必ず 鍵1 → 鍵2 の順）
- **複数の鍵を同時に取るときは `std::lock` / `std::scoped_lock` を使う**
  （デッドロックしない順序で取ってくれる）

鍵を1つしか使わない設計なら、この問題は起きません。
**鍵は少ないほど安全**、というのも設計上の大事な指針です。

## 発展課題4 の解答 ―― 鍵を握ったまま「通知」を出す

```cpp
{
    std::lock_guard<std::mutex> g(mtx);
    queue.push(value);
    notify();          // ← まだ施錠されている
}
```

**正しさの問題はありません。** どちらの書き方でも、結果は同じで正しく動きます。
これは純粋に**効率**の話です。ただし理由を追うには、
「待っているスレッドが何をしているのか」を先に知っておく必要があります。

### 前提：待っているスレッドは「鍵を手放して」眠っている

キューが空のとき、取り出す側のスレッドBは「誰かが入れてくれるまで待つ」状態になります。
このとき B は **鍵を手放してから眠ります**。

なぜ手放すのか。**握ったまま眠ったら、入れる側のスレッドAが鍵を取れず、
永久にキューに入れられないから**です。誰も入れられなければ、B も永久に起きられません。
これは 3-3 で「待つあいだは鍵を開けておく必要がある」と書いた、まさにその話です
（だから `condition_variable` は途中で開け閉めできる `unique_lock` を要求します）。

そしてもう1つ、対になる決まりがあります。

> **眠る前に鍵を手放したのだから、起こされたあとは鍵を取り直さないと再開できない。**

つまり「起こされる」＝「すぐ動ける」ではありません。
起こされたスレッドが最初にやる仕事は、**鍵を取りに行くこと**です。

### 鍵を握ったまま通知すると

この前提のうえで、順を追ってみます。A が入れる側、B が待っている側です。

1. **A** が鍵をかける
2. **A** がキューに入れる
3. **A** が通知を出す ⇒ **B が起こされる**
4. **B** が鍵を取りに行く ⇒ しかし **A がまだ握っている**ので取れない
5. **B** はしかたなく **もう一度眠る** ← ここが無駄
6. **A** がスコープを抜けて解錠する ⇒ **B がもう一度起こされる**
7. **B** がようやく鍵を取れて、再開

B は「起きる → 取れない → 眠る → また起きる」と **2往復**しています。
本来1往復で済むはずのところです。
この様子を **hurry up and wait**（急いで起きたのに、結局また待たされる）と呼びます。

### 先に解錠してから通知すると

```cpp
{
    std::unique_lock<std::mutex> guard(mtx);
    queue.push(value);
    guard.unlock();          // ← 先に解錠してから
}
notify();                    //    通知する
```

1. **A** が解錠する
2. **A** が通知を出す ⇒ **B が起こされる**
3. **B** は鍵をすぐ取れる ⇒ 再開

1往復で済みました。`lock_guard` にはこれができない（途中で解錠する機能がない）ので、
`unique_lock` を使います。**3-3 の「途中で開け閉めできる」がここで生きます。**

### ただし、実際には

最近の実装はこの無駄を減らす工夫（起こす代わりに、鍵の待ち行列へ直接つなぎ替えるなど）を
しており、**差はほとんど測定できないことが多い**です。
鍵を握ったまま通知する書き方は一般的で、実用上まったく問題ありません。
「必ずこう書け」という話ではないので、そこは誤解しないでください。

### この課題の狙い

「どちらが正しいか」ではありません。

> **同じ結果になるコードでも、待たせ方の違いで速さが変わることがある。**

という視点を持ってもらうことです。
そしてもう1つ、**`lock_guard` と `unique_lock` の違いが、
こういう場面で初めて意味を持つ**ということ。3-3 では「途中で開け閉めできる」としか
言えませんでしたが、その機能が要る具体例がこれです。

「正しく動いているのに遅い」原因の多くは、**鍵の掛け方**にあります。
